In [ ]:
import os
from pathlib import Path

# Change working directory to parent directory
import sys
# Locate the repo root (folder containing the `core` package) and make it importable
_root = Path.cwd()
if not (_root / "core").is_dir():
    _root = _root.parent
os.chdir(_root)                       # data/ and outputs/ paths resolve from the repo root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))    # so `core`, `classifiers`, `app`, ... import in any kernel

# Optional: verify
print(Path.cwd())

In [ ]:
# --- vep path bootstrap (auto-added; safe to re-run) ---
import os, sys
from pathlib import Path
def _vep_root():
    cands = []
    _p = globals().get('__vsc_ipynb_file__')           # VSCode exposes the notebook path
    if _p: cands.append(Path(_p).resolve().parent)
    cands.append(Path.cwd().resolve())
    for s in cands:
        for c in [s, *s.parents]:                       # search upward for the repo root
            if (c / 'core').is_dir() and (c / 'classifiers').is_dir():
                return c
    return Path.cwd().resolve()
_root = _vep_root()
os.chdir(_root)                                          # data/ and outputs/ resolve from repo root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))                      # makes core/classifiers/app importable
# --- end vep path bootstrap ---

from core import config
import numpy as np
from core.helpers import get_data
from classifiers.CNN_classifier import CNN1D

print("\n" + "=" * 70)
print("  CNN CLASSIFIER - FULL DATASET TRAINING (ALL DEVICES)")
print("=" * 70)

# Configuration
ALL_DEVICES = ["PRIMA_LE_DA", "PRIMA_RCS_DA", "MP20_LE_DA", "MP20_RCS_LA", "RB20_RCS_LA"]
N_CLASSES = 2  # BC_Only vs BC_and_RGC

# Training parameters from config
EPOCHS = config.EPOCHS
BATCHSIZE = config.BATCHSIZE
LR = config.LR
DROPOUT = config.DROPOUT
L2_LAMBDA = config.L2_LAMBDA
TMAX = getattr(config, 'TMAX', 400)


# Load and combine data from all devices
print(f"\nLoading data from all devices ({N_CLASSES} classes)...")
X_all = []
y_all = []
file_list_all = []

for device in ALL_DEVICES:
    print(f"  Loading {device}...", end=" ")
    X_device, y_device, _, files_device = get_data(device=device, classes=N_CLASSES)
    X_all.extend(X_device)
    y_all.extend(y_device)
    file_list_all.extend(files_device)
    print(f"{len(X_device)} samples")

# Convert to arrays
X_all = np.array(X_all)
y_all = np.array(y_all)

print(f"\nTotal combined dataset: {len(X_all)} samples")
unique_labels, counts = np.unique(y_all, return_counts=True)
for label, count in zip(unique_labels, counts):
    print(f"  {label}: {count}")

print("=" * 70)

# Initialize model
cnn = CNN1D(
    regularization="l2",
    learning_rate=LR,
    l2_lambda=L2_LAMBDA,
    dropout_rate=DROPOUT,
)

# Train and save
model_name = f"combined_all_devices_{N_CLASSES}class"

paths = cnn.fit_and_save(
    X=X_all,
    y=y_all,
    epochs=EPOCHS,
    batch_size=BATCHSIZE,
    model_name=model_name,
    save_dir="app/saved_models",
    device="COMBINED_ALL",
    n_classes=N_CLASSES,
    preprocessing_info={
        "devices": ALL_DEVICES,
        "tmax": TMAX,
        "normalize": True,
        "SNR_filtering": True,
        "dwt_downsampling": True,
        "artifact_removal": False,
        "dwt_level": 4,
    }
)

print("\n" + "=" * 70)
print("  FILES CREATED:")
print("=" * 70)
for key, path in paths.items():
    print(f"  {key}: {path}")
print("=" * 70)

print("\n✓ Model trained on combined dataset from all devices")
print("  Next step: Update paths in app.py with these files")
print("  Then run: python app.py")
print()